Directions:
1) Run All and wait a bit for ultralytics, its dependencies and files
2) Drag and drop video files (avi, mp4, mov) into content->ultralytics->ultralytics->videos
3) Run last cell to use yolo player detection on all files (~1-2 seconds per frame)
4) View detected files in content->ultralytics->ultralytics->all_videos_output

In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git

Cloning into 'ultralytics'...
remote: Enumerating objects: 97412, done.
remote: Counting objects: 100% (961/961), done.
remote: Compressing objects: 100% (427/427), done.
remote: Total 97412 (delta 787), reused 557 (delta 534), pack-reused 96451 (from 2)
Receiving objects: 100% (97412/97412), 53.19 MiB | 12.18 MiB/s, done.
Resolving deltas: 100% (72885/72885), done.


In [ ]:
%pip install ultralytics
import ultralytics
# %pip uninstall torch torchvision torchaudio
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
dependencies = [
    "numpy",
    "matplotlib",
    "opencv-python",
    "pillow",
    "pyyaml",
    "requests",
    "scipy",
    "torch",
    "torchvision",
    "psutil",
    "polars",
    "ultralytics-thop",
]

for dep in dependencies:
    print(f"Checking {dep}:")
    !pip show {dep}
    print("-" * 20)

Checking numpy:
Name: numpy
Version: 2.0.2
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2024, NumPy Developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are
met:

    * Redistributions of source code must retain the above copyright
       notice, this list of conditions and the following disclaimer.

    * Redistributions in binary form must reproduce the above
       copyright notice, this list of conditions and the following
       disclaimer in the documentation and/or other materials provided
       with the distribution.

    * Neither the name of the NumPy Developers nor the names of any
       contributors may be used to endorse or promote products derived
       from this software without specific prior written permission.

THIS SOFTWARE IS PROVID

In [ ]:
import os

# create videos folder to store data.
folder_path = '/content/ultralytics/ultralytics/videos'

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_path}' created.")
else:
    print(f"Folder '{folder_path}' already exists.")

Folder '/content/ultralytics/ultralytics/videos' created.


3/12 Changes:

Run cell below for detections. same instructions as the top description.
- conf = 0.35-0.45, iou = 0.8
- swapped from model.predict to model.track to be able to use bytetrack.yaml for "better" detections
- added batch & workers attribute to have more consistent run times. (detecting X frames at a time)
- video clip for comparison is 14 seconds long.
- Used different ranges of imgsz (image sizes) for detection. [sz640 = ~440 seconds, sz448 = ~382 seconds, sz416 = ~356 seconds, sz320 = ~220 seconds]
- lowering image size lowers run time, but lowers accuracy.
- Personally, sz416 works the best out of all of this so far. It was at sz640 during previous iterations (~10-25 min w/o applied methods). (except for the ball x player one sz992)

4/22 changes:


*   set confidence to 25% due to penalty application
*   Penalties will be applied from outlier check (-40%), area size check (-20%), and region check (-20%)
*   Outlier check: find the mean of all initial detections (>=25%) and apply penalties to outliers
*   Area check: Find the average area (w*l) of all initial detections and apply penalties to detections below the first quartile.
*   Region check: Applies penalties to all detections outside a certain distance of the center of the screen
*   all detections with at least 25% confidence AFTER penalty checks will be the final detections for a given frame.









In [ ]:
from ultralytics import YOLO
import os
import time
import torch # Added for tensor operations in outlier detection
import cv2 # Import OpenCV for manual video processing


# Load a pre-trained YOLO model ('yolov8s.pt' isnt good. get an upgrade [s, m, l])
model = YOLO('yolov8m.pt')

confidence = 0.25 # lowered confidence overall since I'll be applying confidence penalties

# Folder containing videos
video_folder = '/content/ultralytics/ultralytics/videos' # input folder (put videos here)
output_folder = '/content/ultralytics/ultralytics/all_videos_output' # output folder (view videos with detetections here)
os.makedirs(output_folder, exist_ok=True)

# Get list of video files
video_files = [
    os.path.join(video_folder, f)
    for f in os.listdir(video_folder)
    if f.endswith(('.mp4', '.avi', '.mov'))
]

# Check if any videos were found
if not video_files:
    print(f"No video files found in {video_folder}")
else:
    print(f"Found {len(video_files)} videos to look at.")

# instantiate time (this is to show how long this process takes)
start_time = time.time()
# Process each video file individually and save results to a single folder (gonna take rlly long maybe find a way to run them in multiple processes?)
for video_index, video_path in enumerate(video_files):
    print(f"Processing video {video_index + 1}/{len(video_files)}: {os.path.basename(video_path)}")

    # Setup VideoCapture to get video properties
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    output_video_name = os.path.basename(video_path).split('.')[0] + '_filtered_marked.mp4'
    output_video_path = os.path.join(output_folder, output_video_name)
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    # Run YOLOv8 on the current video
    # conf = will box the detected players if they are X% confident.
    # iou = if boxes overlap by X%, keep the one with the highest confidence (maybe set it high cause some sports require close contact)
    results = model.track( # replace predict with track
        source=video_path,
        classes=[0], #class[0] = people
        stream=True,
        imgsz=416, # try lowering image size for detection a bit? faster but tanks accuracy
        save=False,
        persist=True,
        conf=confidence,
        iou=0.8, # if the system thinks it's X percent confident that 2 detected are collided, detect the highest of the two.
        # project=output_folder, # Not needed when save=False
        # name='',  # Not needed when save=False
        exist_ok=True,
        # tracker="bytetrack.yaml", # ultralytics alr has bytetrack.
        cache=True,
        # device=0,
        # vid_stride=4, # process every X seconds. makes it X times faster.
        # persist=True # make sure we keep the changes of vid_stride or else video will speed up X times.
    )

    # --------------------------------------------------------------------------------------------------------

    # Define confidence penalties for different invalid detections
    outlier_penalty = 0.4 # Penalty for detections far from the main cluster
    area_penalty = 0.2    # Penalty for detections with very small area
    region_penalty = 0.2  # Penalty for detections outside the central region

    # Iterate through results (one per frame)
    for i, result in enumerate(results):
        detections = result.boxes
        # Get the original image to draw on (YOLO results.orig_img is RGB, OpenCV expects BGR)
        im_bgr = result.orig_img.copy()

        # Draw the central region for visualization (can remove)
        height, width = result.orig_shape
        region_x1, region_y1 = int(width * 0.2), int(height * 0.2)
        region_x2, region_y2 = int(width * 0.8), int(height * 0.8)
        cv2.rectangle(im_bgr, (region_x1, region_y1), (region_x2, region_y2), (255, 0, 0), 2) # Blue rectangle for region
        cv2.putText(im_bgr, 'Central Region', (region_x1, region_y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)


        if len(detections) > 1:
            # grab coords from detected people
            x_centers = (detections.xyxy[:, 0] + detections.xyxy[:, 2]) / 2
            y_centers = (detections.xyxy[:, 1] + detections.xyxy[:, 3]) / 2
            coords = torch.stack((x_centers, y_centers), dim=1) # (N, 2) tensor of (x, y) centers

            mean_point = torch.mean(coords, dim=0)

            # Draw mean point (can remove. (just visualization))
            cv2.circle(im_bgr, (int(mean_point[0]), int(mean_point[1])), 5, (0, 255, 255), -1) # Yellow circle for mean point
            cv2.putText(im_bgr, 'Mean', (int(mean_point[0]) + 10, int(mean_point[1]) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

            # Outlier Filter
            distances = torch.linalg.norm(coords - mean_point, dim=1)
            median_dist = torch.median(distances)
            mad_dist = torch.median(torch.abs(distances - median_dist)) # Median Absolute Deviation (MAD)
            # Outlier points are those whose distances are significantly larger than the median distance
            outlier_threshold = median_dist + 0.5 * (mad_dist / 0.6745)
            outlier_indices = distances > outlier_threshold # True if outlier (FAIL), False if NOT outlier (PASS)

            # Area Filter
            areas = (detections.xyxy[:, 2] - detections.xyxy[:, 0]) * (detections.xyxy[:, 3] - detections.xyxy[:, 1])
            area_indices = areas > 0.5 * torch.median(areas) # True if good area (PASS)

            # Region Filter
            region_indices = (
                (x_centers > width * 0.1) & (x_centers < width * 0.9) &
                (y_centers > height * 0.1) & (y_centers < height * 0.9)
            ) # True if within region (PASS)

            # Create a mutable copy of the confidence tensor before modifying
            conf_scores_modified = detections.conf.clone()

            # Apply penalties based on different filters (if FAIL)
            # Apply outlier penalty
            conf_scores_modified[outlier_indices] = torch.clamp(conf_scores_modified[outlier_indices] - outlier_penalty, min=0.01)
            # Apply area penalty
            # conf_scores_modified[area_indices] = torch.clamp(conf_scores_modified[area_indices] + area_penalty, min=1.0)
            conf_scores_modified[~area_indices] = torch.clamp(conf_scores_modified[~area_indices] - area_penalty, min=0.01)
            # Apply region penalty
            # conf_scores_modified[region_indices] = torch.clamp(conf_scores_modified[region_indices] + region_penalty, min=1.0)
            conf_scores_modified[~region_indices] = torch.clamp(conf_scores_modified[~region_indices] - region_penalty, min=0.01)

            # Count people after applying all potential confidence penalties
            num_people = len(conf_scores_modified[conf_scores_modified >= confidence])

            # Drawing loop: Draw based on the final filtering (valid_indices and conf_scores_modified)
            for i in range(len(detections)):
                box = detections.xyxy[i].cpu().numpy().astype(int)
                x1, y1, x2, y2 = box[0], box[1], box[2], box[3]
                # original_confidence = detections.conf[idx].item() # No longer needed for drawing decision
                modified_confidence = conf_scores_modified[i].item()

                # this is for visual debug (can remove)
                extra_info = ""
                if outlier_indices[i]: # Check if this specific detection triggered the outlier penalty (O = outlier, A = area, R = region)
                    extra_info += "O"
                if !area_indices[i]:
                    extra_info += "A"
                if !region_indices[i]:
                    extra_info += "R"

                # Draw detections for valid detections that meet final threshold
                if modified_confidence >= confidence:
                    cv2.rectangle(im_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2) # Green
                    label = f"P {modified_confidence:.2f}{extra_info}" # Show modified confidence + outlier info
                    cv2.putText(im_bgr, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                # else:
                    # # Draw red rectangle for filtered/non-counted detections (this was for debugging)
                    # cv2.rectangle(im_bgr, (x1, y1), (x2, y2), (0, 0, 255), 2) # Red
                    # label = f"Filtered {modified_confidence:.2f}{extra_info}"
                    # cv2.putText(im_bgr, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        else:
            num_people = len(detections[detections.conf >= confidence])

            # Drawing loop for 0 or 1 detection case (based only on original confidence threshold)
            for i in range(len(detections)):
                box = detections.xyxy[i].cpu().numpy().astype(int)
                x1, y1, x2, y2 = box[0], box[1], box[2], box[3]
                original_confidence = detections.conf[i].item()

                extra_info = ""

                if detections.conf[i] >= confidence:
                    cv2.rectangle(im_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2) # Green
                    label = f"P {original_confidence:.2f}{extra_info}"
                    cv2.putText(im_bgr, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        print(f"  Video {video_index + 1}, Frame {i+1}: Detected {num_people} people")
        out.write(im_bgr) # Write the modified frame to the output video

    cap.release() # Release VideoCapture for the current video
    out.release() # Release VideoWriter for the current video
    print(f"Processed video with filtered detections marked saved to {output_video_path}")

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken: {total_time} second(s) ({total_time/60} minute(s)) to view {len(video_files)} files.")

Found 1 videos to look at.
Processing video 1/1: testing_video.mp4

video 1/1 (frame 1/199) /content/ultralytics/ultralytics/videos/testing_video.mp4: 256x416 10 persons, 379.9ms
  Video 1, Frame 10: Detected 7 people
video 1/1 (frame 2/199) /content/ultralytics/ultralytics/videos/testing_video.mp4: 256x416 10 persons, 376.7ms
  Video 1, Frame 10: Detected 7 people
video 1/1 (frame 3/199) /content/ultralytics/ultralytics/videos/testing_video.mp4: 256x416 10 persons, 381.1ms
  Video 1, Frame 10: Detected 6 people
video 1/1 (frame 4/199) /content/ultralytics/ultralytics/videos/testing_video.mp4: 256x416 10 persons, 402.0ms
  Video 1, Frame 10: Detected 7 people
video 1/1 (frame 5/199) /content/ultralytics/ultralytics/videos/testing_video.mp4: 256x416 10 persons, 522.9ms
  Video 1, Frame 10: Detected 7 people
video 1/1 (frame 6/199) /content/ultralytics/ultralytics/videos/testing_video.mp4: 256x416 10 persons, 623.7ms
  Video 1, Frame 10: Detected 6 people
video 1/1 (frame 7/199) /content